# Fine-Tuning Llama 3.1 8B per la ricerca mirata di Ricette
Questo notebook configura il processo di **Fine-Tuning (QLoRA)** per specializzare Llama 3.1 8B nella generazione di query di ricerca altamente ottimizzate per le ricette italiane. 

Il processo si divide in:
1. Setup dell'ambiente (Librerie necessarie per addestrare in locale in 4-bit)
2. Generazione di un dataset di addestramento
3. Caricamento del modello base in 4-bit (per risparmiare memoria)
4. Addestramento (SFT) e misurazione della Loss
5. Inferenza di test
6. Esportazione (per usarlo su Ollama)

In [ ]:
import sys
!{sys.executable} -m pip uninstall -y torch torchvision
!{sys.executable} -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132
!{sys.executable} -m pip install -q -U transformers peft trl bitsandbytes datasets

Found existing installation: torch 2.12.0+cu126
Uninstalling torch-2.12.0+cu126:
  Successfully uninstalled torch-2.12.0+cu126
Found existing installation: torchvision 0.27.0+cu126
Uninstalling torchvision-0.27.0+cu126:
  Successfully uninstalled torchvision-0.27.0+cu126


You can safely remove it manually.
You can safely remove it manually.


Looking in indexes: https://download.pytorch.org/whl/cu132


ERROR: Could not find a version that satisfies the requirement torchaudio (from versions: none)
ERROR: No matching distribution found for torchaudio


In [5]:

%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [5]:
import sys
import torch
print("Python Executable:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA Version built with:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

Python Executable: c:\Users\xolor\Miniconda3\python.exe
Torch version: 2.12.0+cu126
CUDA Version built with: 12.6
CUDA Available: True
cuda


In [7]:
import os
os.environ["PYTHONUTF8"] = "1"

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# 2. Creazione Dataset di Addestramento
# Creiamo dataset (instruction -> output) con esempi di query perfette per DDG in base all'intento.
data = [
    {"instruction": "Devo preparare una carbonara, cosa cerco?", "output": "ricetta originale carbonara romana guanciale pecorino"},
    {"instruction": "Voglio fare il tiramisù classico.", "output": "ricetta tiramisù tradizionale savoiardi mascarpone caffè"},
    {"instruction": "Fammi cercare i passaggi per la vera pizza margherita.", "output": "pizza margherita napoletana impasto preparazione originale"},
    {"instruction": "Cerca la cacio e pepe.", "output": "ricetta cacio e pepe tradizionale romana ingredienti"},
    {"instruction": "Mi serve la ricetta del ragù alla bolognese.", "output": "ragù alla bolognese originale ricetta depositata bologna"},
    {"instruction": "Preparami il pesto alla genovese.", "output": "pesto alla genovese ricetta tradizionale mortaio basilico DOP"},
    # Aggiungi qui molti altri esempi per un buon fine tuning...
]

# Formattazione per Llama 3 Instuct
def format_prompt(sample):
    # Riprende il prompt di sistema usato da llama-3.1
    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nSei un assistente per la ricerca di ricette italiane. Il tuo scopo è tradurre la richiesta dell'utente nell'esatta e migliore stringa di ricerca da inviare a un motore di ricerca.<|eot_id|><|start_header_id|>user<|end_header_id|>\n{sample['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n{sample['output']}<|eot_id|>"
    return {"text": prompt}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompt)
print("Dataset preparato:", dataset[0]["text"])

RuntimeError: Failed to import trl.trainer.sft_trainer because of the following error (look up to see its traceback):
Could not import module 'AutoProcessor'. Are this object's requirements defined correctly?

In [1]:
# 3. Caricamento Modello e Tokenizer
import os
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


# Carica le variabili di ambiente dal file .env (assicurati di aver inserito HF_TOKEN lì dentro)
load_dotenv()
hf_token = os.environ.get("HF_TOKEN")

# Effettua il login globale su Hugging Face
if hf_token:
    login(token=hf_token)
else:
    print("ATTENZIONE: HF_TOKEN non trovato. Inseriscilo nel file .env (es. HF_TOKEN=hf_...)")

# Usiamo Unsloth o i repository della community che pesano meno e non richiedono token HF se usati correttamente.
model_id = "unsloth/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
# Impostiamo il padding token (Llama 3 non ce l'ha di default)
tokenizer.pad_token = tokenizer.eos_token

# Configurazione della quantizzazione a 4-bit (bitsandbytes)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True # Aggiunto per permettere l'offload su CPU/RAM se la VRAM si riempie
)

# Caricamento del modello
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)
model = prepare_model_for_kbit_training(model)

# Configurazione PEFT (LoRA)
peft_config = LoraConfig(
    r=16,          # Rank della matrice
    lora_alpha=32, # Parametro di scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

c:\Users\xolor\Miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
c:\Users\xolor\Miniconda3\Lib\site-packages\torch\cuda\__init__.py:384: UserWarning: Found GPU0 NVIDIA GeForce RTX 5070 which is of compute capability (CC) 12.0.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 5.0 which supports hardware CC >=5.0,<6.0 except {5.3}
- 6.0 which supports hardware CC >=6.0,<7.0 except {6.2}
- 6.1 which supports hardware CC >=6.1,<7.0 except {6.2}
- 7.0 which supports hardware CC >=7.0,<8.0 except {7.2}
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardw

RuntimeError: We encountered some issues during automatic conversion of the weights. For details look at the `CONVERSION` entries of the above report!

In [ ]:
# 4. Addestramento e Valutazione Metriche

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    logging_steps=2,          # Quanti step saltare prima di loggare le metriche (loss)
    learning_rate=2e-4,
    fp16=True,                # Usa Half-Precision se GPU compatibile per velocizzare
    max_steps=50,             # Aumenta se il dataset è molto grande (almeno 100-300 epoch virtuali)
    evaluation_strategy="no", # Cambia a "steps" se hai diviso test_dataset
    save_strategy="steps",
    save_steps=25,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=256,       # Dimensione della finestra. Breve dato il contesto delle queries
    tokenizer=tokenizer,
    args=training_args,
)

# Avvia Fine Tuning
print("=== Avvio Addestramento ===")
metrics = trainer.train()

print("\n--- Metriche ---")
print(metrics.metrics)

In [ ]:
# 5. Esempio di Inferenza Post FineTuning
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
prompt_test = "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nSei un assistente per la ricerca di ricette italiane. Il tuo scopo è tradurre la richiesta dell'utente nell'esatta e migliore stringa di ricerca da inviare a un motore di ricerca.<|eot_id|><|start_header_id|>user<|end_header_id|>\nSto cercando la ricetta delle lasagne al forno.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
inputs = tokenizer(prompt_test, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 6. Salvare e Integrare con Ollama
Se il modello soddisfa le metriche, devi fonderlo (*merge*) per ottenere l'aggiornamento definitivo, dopodiché potrai importarlo nativamente nel tuo ambiente locale.
Poiché l'esportazione verso formato `GGUF` (il formato letto da Ollama) richiede script C++ ausiliari della libreria *llama.cpp*, il procedimento in Python nativo puro può essere complesso.

In alternativa:
Puoi salvare gli adapter su disco:
```python
model.save_pretrained("llama3-recipe-adapter")
```
Dopodiché esportare l'intero modello fuso per Ollama tramite librerie di helper come `unsloth` integrato, che permettono di fare `model.save_pretrained_gguf("model", tokenizer)`. 

Una volta ottenuto il file `tuomodello.gguf`, userai un `Modelfile` del genere per Ollama:
```dockerfile
FROM ./tuomodello.gguf
```
Da terminale digiterai:
`ollama create llama-ricette -f Modelfile`

E nel main `blog_agent.py` potrai aggiornarlo in modo semplicissimo:
`llm = ChatOllama(model="llama-ricette", temperature=0.5)`